# 09 · Demo: the student's free-road predictions on unseen frames

Runs a trained student (a notebook-07 run) on consecutive frames of a drive and writes, per frame, the predicted maps (free road D_free, occupancy, unknown) and an overlay: the camera image with the predicted free road (green) and occupied cells (red) drawn at ground level, next to the bird's-eye-view map. The frames are joined into `demo.mp4` and `demo.gif`.

**No ground truth is used**: the model sees only the image, the scan and the calibration, as at inference. Default frames: drive 0000's frames 10354-10653 (30 s), in the **test** part of the starter split, so the model never trained on them and they never chose anything. Starter-phase model: a demonstration, not a result.

## Before you run

1. **Settings → Accelerator:** GPU T4 (CPU works, slower). **Internet:** On (code clone, DINOv2 weights).
2. **Add Input:** the notebook-07 output holding `RUN` (e.g. `ocgf-runs-e0-s0`), drive 0000's full data-prep output and the KITTI-360 starter. Object runs (E1/E2) also need `ocgf-mtrans-objects-v1`.
3. **Save Version → Save & Run All** (about 5 minutes for 300 frames).

In [ ]:
from pathlib import Path

REPO_URL  = "https://github.com/GNaveenkumar614/OCGF.git"
GIT_REF   = "p0/skeleton-geometry"       # branch, tag or commit (main once merged)
RUN       = "dev_e0_concat_s0"           # a notebook-07 run folder (attached as input)
DRIVE     = "2013_05_28_drive_0000_sync"
FRAMES    = "10354-10653"                # consecutive frames, test part of kitti360_d0000_dev_v1 (never trained on)
EVERY     = 1                            # every stored frame (10 per second on drive 0000)
MAX_FRAMES = 300
PERTURB   = ""                           # optional, e.g. "rings=4" to show a 16-beam LiDAR
FPS       = 10
INPUT_ROOT = Path("/kaggle/input")
OUT_DIR   = Path("/kaggle/working/demo")
STAGE_DIR = Path("/kaggle/temp/stage")
THREADS   = 32
CODE_DIR  = Path("/kaggle/temp/OCGF")

## Code

A fresh clone of the repo at `GIT_REF`, installed in editable mode.

In [ ]:
import os, shutil, subprocess, sys

def sh(cmd, cwd=None):
    # run a command, stream its output into this notebook, and stop Run All if it fails
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")

CODE_DIR.parent.mkdir(parents=True, exist_ok=True)
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)            # fresh clone, so the run matches GIT_REF exactly
sh(f'git clone -q {REPO_URL} "{CODE_DIR}"')
sh(f'git -C "{CODE_DIR}" checkout -q {GIT_REF}')
sh(f'git -C "{CODE_DIR}" log -1 --date=short --format="code at commit %h (%cd): %s"')
sh(f'{sys.executable} -m pip install -q -e "{CODE_DIR}"')
sys.path.insert(0, str(CODE_DIR))
for name in [m for m in sys.modules if m == "ocgf" or m.startswith("ocgf.")]:
    del sys.modules[name]

from ocgf.utils.provenance import environment
print(environment())

## Predict

Finds the run among the inputs and writes the maps, the overlay frames and the video.

In [ ]:
from ocgf.datasets.locate import find_runs

hits = find_runs(INPUT_ROOT, RUN)
assert hits, f"run {RUN} not found: attach its notebook-07 output (Add Input)"
for other in hits[1:]:
    print(f"WARNING: {RUN} is also attached at {other} (not used)")
sh(f'{sys.executable} scripts/predict.py --run "{hits[0]}" --input-root "{INPUT_ROOT}" --sequence {DRIVE} '
   f'--frames {FRAMES} --every {EVERY} --max-frames {MAX_FRAMES} --objects "{INPUT_ROOT}" --out "{OUT_DIR}" '
   f'--video --fps {FPS} --stage-dir "{STAGE_DIR}" --threads {THREADS}'
   + (f' --perturb "{PERTURB}"' if PERTURB else ""), cwd=CODE_DIR)

## The result

The animated GIF and two single frames. Download `demo.mp4` (or `demo.gif`) from the output.

In [ ]:
from IPython.display import Image, display

frames = sorted((OUT_DIR / "frames").glob("*.png"))
print(f"{len(frames)} frames; videos: {[p.name for p in OUT_DIR.glob('demo.*')]}")
if (OUT_DIR / "demo.gif").exists():
    display(Image(filename=str(OUT_DIR / "demo.gif")))
for png in frames[:: max(1, len(frames) // 2)][:2]:
    display(Image(filename=str(png)))
sh(f'du -sh "{OUT_DIR}"/*')

## After the run

The output holds `demo/demo.mp4`, `demo/demo.gif`, `demo/frames/<frame>.png` and `demo/maps/<frame>.npz` (p_road, p_occ, p_unk, d_free). To compare runs, run again with another `RUN` (e.g. `dev_e0_lidar_s0`) or a `PERTURB`.